In [4]:
import requests
import pandas as pd
import re
import time
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

In [5]:
class MeroLaganiHistoricalScraper:
    """
    MeroLagani Historical Data Scraper (1 Year)
    """

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Connection': 'keep-alive',
        })
        self.all_data = []

    def clean_number(self, text):
        """Handle numbers with K/M/B suffixes"""
        if not text or text in ['', '-', 'N/A', 'null', None, '--', 'n/a']:
            return None
        try:
            text_str = str(text).strip().upper()
            multiplier = 1
            if text_str.endswith('K'):
                multiplier = 1000
                text_str = text_str[:-1]
            elif text_str.endswith('M'):
                multiplier = 1000000
                text_str = text_str[:-1]
            elif text_str.endswith('B'):
                multiplier = 1000000000
                text_str = text_str[:-1]
            cleaned = re.sub(r'[^\d.-]', '', text_str)
            if cleaned and cleaned not in ['-', '.', '']:
                return float(cleaned) * multiplier
        except:
            pass
        return None

    def get_trading_dates(self, start_date, end_date):
        """Generate list of trading days (skip Fri & Sat in Nepal + major holidays)"""
        dates = []
        current = start_date
        holidays = ['2024-01-01', '2024-04-13', '2024-10-24', '2024-11-12']
        while current <= end_date:
            if current.weekday() not in [4, 5]:  # Fri, Sat
                if current.strftime('%Y-%m-%d') not in holidays:
                    dates.append(current)
            current += timedelta(days=1)
        return dates

    def scrape_merolagani_historical(self, date):
        """Scrape MeroLagani data for a given date"""
        date_str = date.strftime('%Y-%m-%d')
        urls = [
            f"https://merolagani.com/LatestMarket.aspx?date={date_str}",
            f"https://www.merolagani.com/LatestMarket.aspx?date={date_str}"
        ]

        for url in urls:
            try:
                response = self.session.get(url, timeout=20)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    tables = soup.find_all('table')
                    for table in tables:
                        rows = table.find_all('tr')
                        if len(rows) > 5:
                            stocks = []
                            header_found = False
                            for row in rows:
                                cells = [c.get_text(strip=True) for c in row.find_all(['td', 'th'])]
                                if not cells:
                                    continue
                                if any("symbol" in c.lower() for c in cells):
                                    header_found = True
                                    continue
                                if header_found and len(cells) >= 7:
                                    symbol = None
                                    if re.match(r'^[A-Z]{3,10}$', cells[0].strip()):
                                        symbol = cells[0].strip()
                                    if symbol:
                                        stock = {
                                            'date': date_str,
                                            'symbol': symbol,
                                            'close': self.clean_number(cells[1]),
                                            'change_percent': self.clean_number(cells[2]),
                                            'open': self.clean_number(cells[3]),
                                            'high': self.clean_number(cells[4]),
                                            'low': self.clean_number(cells[5]),
                                            'volume': self.clean_number(cells[6]),
                                            'source': 'merolagani_historical',
                                            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                                        }
                                        if stock['close'] is not None:
                                            stocks.append(stock)
                            if stocks:
                                return stocks
            except:
                continue
        return None

    def scrape_year_data(self, start_date=None, end_date=None, delay_seconds=2):
        """Fetch 1 year of data"""
        if not end_date:
            end_date = datetime.now()
        if not start_date:
            start_date = end_date - timedelta(days=365)

        trading_dates = self.get_trading_dates(start_date, end_date)
        for i, date in enumerate(trading_dates):
            print(f"[{i+1}/{len(trading_dates)}] {date.strftime('%Y-%m-%d')}")
            day_data = self.scrape_merolagani_historical(date)
            if day_data:
                self.all_data.extend(day_data)
            time.sleep(delay_seconds)
        return self.get_dataframe()

    def get_dataframe(self):
        """Return cleaned DataFrame with standardized schema"""
        if not self.all_data:
            return pd.DataFrame()
        df = pd.DataFrame(self.all_data)
        df['date'] = pd.to_datetime(df['date'], errors="coerce")
        df = df.drop_duplicates(subset=['symbol', 'date'])
        df = df.sort_values(['date', 'symbol']).reset_index(drop=True)

        # Ensure expected schema
        expected = ["date", "symbol", "open", "high", "low", "close",
                    "volume", "change_percent", "source", "timestamp"]
        df = df[[c for c in expected if c in df.columns]]
        return df

    def save_data(self, df, filename=None):
        """Save standardized historical data"""
        if df.empty:
            print("No data to save")
            return None
        if not filename:
            start_date = df['date'].min().strftime('%Y%m%d')
            end_date = df['date'].max().strftime('%Y%m%d')
            filename = f"MeroLagani_Historical_{start_date}_to_{end_date}.csv"
        df.to_csv(filename, index=False)
        print(f"Data saved to {filename}")
        return filename

In [6]:
# ------------------ USAGE ------------------
if __name__ == "__main__":
    scraper = MeroLaganiHistoricalScraper()
    
    # Define exact date range
    start_date = datetime(2024, 8, 25)
    end_date   = datetime(2025, 8, 25)

    # Fetch with custom range
    df = scraper.scrape_year_data(start_date=start_date, end_date=end_date, delay_seconds=3)
    
    if not df.empty:
        print(f"Collected {len(df)} records from {start_date.date()} to {end_date.date()}")
        scraper.save_data(df, filename="D:/TRADING_SYSTEM/backend/core/data/master_data.csv")
        print(df.head())
    else:
        print("Failed to fetch historical data")

[1/260] 2024-08-25
[2/260] 2024-08-26
[3/260] 2024-08-27
[4/260] 2024-08-28
[5/260] 2024-08-29
[6/260] 2024-09-01
[7/260] 2024-09-02
[8/260] 2024-09-03
[9/260] 2024-09-04
[10/260] 2024-09-05
[11/260] 2024-09-08
[12/260] 2024-09-09
[13/260] 2024-09-10
[14/260] 2024-09-11
[15/260] 2024-09-12
[16/260] 2024-09-15
[17/260] 2024-09-16
[18/260] 2024-09-17
[19/260] 2024-09-18
[20/260] 2024-09-19
[21/260] 2024-09-22
[22/260] 2024-09-23
[23/260] 2024-09-24
[24/260] 2024-09-25
[25/260] 2024-09-26
[26/260] 2024-09-29
[27/260] 2024-09-30
[28/260] 2024-10-01
[29/260] 2024-10-02
[30/260] 2024-10-03
[31/260] 2024-10-06
[32/260] 2024-10-07
[33/260] 2024-10-08
[34/260] 2024-10-09
[35/260] 2024-10-10
[36/260] 2024-10-13
[37/260] 2024-10-14
[38/260] 2024-10-15
[39/260] 2024-10-16
[40/260] 2024-10-17
[41/260] 2024-10-20
[42/260] 2024-10-21
[43/260] 2024-10-22
[44/260] 2024-10-23
[45/260] 2024-10-27
[46/260] 2024-10-28
[47/260] 2024-10-29
[48/260] 2024-10-30
[49/260] 2024-10-31
[50/260] 2024-11-03
[51/260] 

In [1]:
#backend/scripts/merolagani_historical.py
import requests
import pandas as pd
import re
import time
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

class MeroLaganiHistoricalScraper:
    """
    MeroLagani Historical Data Scraper (1 Year)
    """

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Connection': 'keep-alive',
        })
        self.all_data = []

    def clean_number(self, text):
        """Handle numbers with K/M/B suffixes"""
        if not text or text in ['', '-', 'N/A', 'null', None, '--', 'n/a']:
            return None
        try:
            text_str = str(text).strip().upper()
            multiplier = 1
            if text_str.endswith('K'):
                multiplier = 1000
                text_str = text_str[:-1]
            elif text_str.endswith('M'):
                multiplier = 1000000
                text_str = text_str[:-1]
            elif text_str.endswith('B'):
                multiplier = 1000000000
                text_str = text_str[:-1]
            cleaned = re.sub(r'[^\d.-]', '', text_str)
            if cleaned and cleaned not in ['-', '.', '']:
                return float(cleaned) * multiplier
        except:
            pass
        return None

    def get_trading_dates(self, start_date, end_date):
        """Generate list of trading days (skip Fri & Sat in Nepal + major holidays)"""
        dates = []
        current = start_date
        holidays = ['2024-01-01', '2024-04-13', '2024-10-24', '2024-11-12']
        while current <= end_date:
            if current.weekday() not in [4, 5]:  # Fri, Sat
                if current.strftime('%Y-%m-%d') not in holidays:
                    dates.append(current)
            current += timedelta(days=1)
        return dates

    def scrape_merolagani_historical(self, date):
        """Scrape MeroLagani data for a given date"""
        date_str = date.strftime('%Y-%m-%d')
        urls = [
            f"https://merolagani.com/LatestMarket.aspx?date={date_str}",
            f"https://www.merolagani.com/LatestMarket.aspx?date={date_str}"
        ]

        for url in urls:
            try:
                response = self.session.get(url, timeout=20)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    tables = soup.find_all('table')
                    for table in tables:
                        rows = table.find_all('tr')
                        if len(rows) > 5:
                            stocks = []
                            header_found = False
                            for row in rows:
                                cells = [c.get_text(strip=True) for c in row.find_all(['td', 'th'])]
                                if not cells:
                                    continue
                                if any("symbol" in c.lower() for c in cells):
                                    header_found = True
                                    continue
                                if header_found and len(cells) >= 7:
                                    symbol = None
                                    if re.match(r'^[A-Z]{3,10}$', cells[0].strip()):
                                        symbol = cells[0].strip()
                                    if symbol:
                                        stock = {
                                            'date': date_str,
                                            'symbol': symbol,
                                            'close': self.clean_number(cells[1]),
                                            'change_percent': self.clean_number(cells[2]),
                                            'open': self.clean_number(cells[3]),
                                            'high': self.clean_number(cells[4]),
                                            'low': self.clean_number(cells[5]),
                                            'volume': self.clean_number(cells[6]),
                                            'source': 'merolagani_historical',
                                            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                                        }
                                        if stock['close'] is not None:
                                            stocks.append(stock)
                            if stocks:
                                return stocks
            except Exception as e:
                print(f"Error: {str(e)}")
                continue
        return None

    def scrape_year_data(self, start_date=None, end_date=None, delay_seconds=2):
        """Fetch 1 year of data"""
        if not end_date:
            end_date = datetime.now()
        if not start_date:
            start_date = end_date - timedelta(days=365)

        trading_dates = self.get_trading_dates(start_date, end_date)
        print(f"\n{'='*80}")
        print(f"STARTING HISTORICAL DATA COLLECTION")
        print(f"{'='*80}")
        print(f"Date Range: {start_date.date()} to {end_date.date()}")
        print(f"Trading Days: {len(trading_dates)}")
        print(f"Estimated Time: ~{len(trading_dates) * delay_seconds / 60:.1f} minutes")
        print(f"{'='*80}\n")
        
        success_count = 0
        fail_count = 0
        
        for i, date in enumerate(trading_dates):
            print(f"[{i+1}/{len(trading_dates)}] Scraping {date.strftime('%Y-%m-%d')}...", end=' ')
            day_data = self.scrape_merolagani_historical(date)
            if day_data:
                self.all_data.extend(day_data)
                print(f"✓ Got {len(day_data)} stocks")
                success_count += 1
            else:
                print("✗ No data")
                fail_count += 1
            time.sleep(delay_seconds)
        
        print(f"\n{'='*80}")
        print(f"COLLECTION COMPLETE")
        print(f"Total Records: {len(self.all_data)}")
        print(f"Success: {success_count} days | Failed: {fail_count} days")
        print(f"{'='*80}\n")
        
        return self.get_dataframe()

    def get_dataframe(self):
        """Return cleaned DataFrame with standardized schema"""
        if not self.all_data:
            return pd.DataFrame()
        df = pd.DataFrame(self.all_data)
        df['date'] = pd.to_datetime(df['date'], errors="coerce")
        df = df.drop_duplicates(subset=['symbol', 'date'])
        df = df.sort_values(['date', 'symbol']).reset_index(drop=True)

        # Ensure expected schema
        expected = ["date", "symbol", "open", "high", "low", "close",
                    "volume", "change_percent", "source", "timestamp"]
        df = df[[c for c in expected if c in df.columns]]
        return df

    def save_data(self, df, filename=None):
        """Save standardized historical data"""
        if df.empty:
            print("❌ No data to save")
            return None
        if not filename:
            start_date = df['date'].min().strftime('%Y%m%d')
            end_date = df['date'].max().strftime('%Y%m%d')
            filename = f"MeroLagani_Historical_{start_date}_to_{end_date}.csv"
        df.to_csv(filename, index=False)
        print(f"\n✅ Data saved to {filename}")
        print(f"   Total records: {len(df)}")
        print(f"   Date range: {df['date'].min().date()} to {df['date'].max().date()}")
        print(f"   Unique symbols: {df['symbol'].nunique()}")
        print(f"   Unique dates: {df['date'].nunique()}")
        return filename

# ------------------ TEST MODE ------------------
if __name__ == "__main__":
    scraper = MeroLaganiHistoricalScraper()
    
    print("\n" + "="*80)
    print("NEPSE HISTORICAL DATA SCRAPER - TEST MODE")
    print("="*80)
    
    # TEST MODE: Just 10 days to verify scraper works
    end_date = datetime.now()
    start_date = end_date - timedelta(days=10)
    
    print(f"Test Mode: Collecting 10 days of data")
    print(f"Start Date: {start_date.date()}")
    print(f"End Date: {end_date.date()}")
    print("="*80)

    # Fetch with test range
    df = scraper.scrape_year_data(
        start_date=start_date, 
        end_date=end_date, 
        delay_seconds=3  # Slower to avoid blocking
    )
    
    if not df.empty:
        # Save test data first
        test_file = "test_historical.csv"
        scraper.save_data(df, filename=test_file)
        
        print("\n" + "="*80)
        print("VALIDATION CHECKS")
        print("="*80)
        
        # Check if data is real (prices should vary)
        sample_symbols = df['symbol'].unique()[:3]  # Check first 3 symbols
        
        all_valid = True
        for symbol in sample_symbols:
            symbol_data = df[df['symbol'] == symbol].sort_values('date')
            unique_prices = symbol_data['close'].nunique()
            total_days = len(symbol_data)
            
            print(f"\n{symbol}:")
            print(f"  Days of data: {total_days}")
            print(f"  Unique prices: {unique_prices}")
            print(f"  Close prices: {symbol_data['close'].tolist()}")
            
            if unique_prices == 1 and total_days > 1:
                print(f"  ⚠️  WARNING: All prices are identical! Data might be corrupted.")
                all_valid = False
            elif unique_prices > 1:
                print(f"  ✅ Prices vary correctly (real data)")
            else:
                print(f"  ℹ️  Only 1 day of data")
        
        print("\n" + "="*80)
        if all_valid:
            print("✅ VALIDATION PASSED - Data looks good!")
            print("\nNext steps:")
            print("1. Run full year scraping by changing days to 365")
            print("2. Copy test_historical.csv to master_data.csv location")
            print("3. Or uncomment the PRODUCTION MODE below")
        else:
            print("⚠️  VALIDATION FAILED - Data appears corrupted")
            print("\nPossible issues:")
            print("1. MeroLagani website might be blocking requests")
            print("2. Website structure might have changed")
            print("3. Try using alternative data source (NEPSE Alpha API)")
        print("="*80 + "\n")
        
    else:
        print("\n" + "="*80)
        print("❌ FAILED - No historical data collected")
        print("="*80)
        print("\nTroubleshooting:")
        print("1. Check internet connection")
        print("2. Try accessing https://merolagani.com manually")
        print("3. Website might be temporarily down")
        print("4. Consider using alternative data source")
        print("="*80 + "\n")

    # PRODUCTION MODE (uncomment after test passes)
    """
    print("\n" + "="*80)
    print("SWITCHING TO PRODUCTION MODE - FULL YEAR")
    print("="*80)
    
    scraper_prod = MeroLaganiHistoricalScraper()
    end_date_prod = datetime.now()
    start_date_prod = end_date_prod - timedelta(days=365)
    
    df_prod = scraper_prod.scrape_year_data(
        start_date=start_date_prod,
        end_date=end_date_prod,
        delay_seconds=2
    )
    
    if not df_prod.empty:
        output_path = "D:/Trading_system/backend/core/data/master_data.csv"
        scraper_prod.save_data(df_prod, filename=output_path)
        print("\n✅ SUCCESS - Full year data saved to master_data.csv")
    """


NEPSE HISTORICAL DATA SCRAPER - TEST MODE
Test Mode: Collecting 10 days of data
Start Date: 2025-11-14
End Date: 2025-11-24

STARTING HISTORICAL DATA COLLECTION
Date Range: 2025-11-14 to 2025-11-24
Trading Days: 7
Estimated Time: ~0.3 minutes

[1/7] Scraping 2025-11-16... ✓ Got 286 stocks
[2/7] Scraping 2025-11-17... ✓ Got 286 stocks
[3/7] Scraping 2025-11-18... ✓ Got 286 stocks
[4/7] Scraping 2025-11-19... ✓ Got 286 stocks
[5/7] Scraping 2025-11-20... ✓ Got 286 stocks
[6/7] Scraping 2025-11-23... ✓ Got 286 stocks
[7/7] Scraping 2025-11-24... ✓ Got 286 stocks

COLLECTION COMPLETE
Total Records: 2002
Success: 7 days | Failed: 0 days


✅ Data saved to test_historical.csv
   Total records: 2002
   Date range: 2025-11-16 to 2025-11-24
   Unique symbols: 286
   Unique dates: 7

VALIDATION CHECKS

ACLBSL:
  Days of data: 7
  Unique prices: 1
  Close prices: [1010.2, 1010.2, 1010.2, 1010.2, 1010.2, 1010.2, 1010.2]
  ⚠️  WARNING: All prices are identical! Data might be corrupted.

ADBL:
  Day

In [2]:
import pandas as pd

df = pd.read_csv("test_historical.csv")
print(f"Total records: {len(df)}")
print(f"Unique dates: {df['date'].nunique()}")
print(f"Unique symbols: {df['symbol'].nunique()}")

# Check one symbol
aclbsl = df[df['symbol'] == 'ACLBSL']
print(f"\nACLBSL prices: {aclbsl['close'].tolist()}")

Total records: 2002
Unique dates: 7
Unique symbols: 286

ACLBSL prices: [1010.2, 1010.2, 1010.2, 1010.2, 1010.2, 1010.2, 1010.2]
